# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/G-Yadidhya/notebook/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane: Refresh / Content Opportunity Scoring.** I frame this as a **ranking/scoring** task: produce a review-priority score for each content item so an editor can decide which pages to inspect first.

- **Decision:** Which pages should the content team review first for a possible refresh, expansion, protection, pruning, or monitoring?
- **User and action:** An editor reviews a ranked queue, checks the reason codes and page context, then decides whether any change is appropriate.
- **Cost of a wrong call:** A false positive wastes limited editorial time; a false negative can leave a genuinely weakening but valuable page unattended. A score is a shortlist, not an automatic instruction to change a page.
- **ML-loop view:** define the decision and unit → build safe features → compare a simple rule baseline with a learned score → evaluate on held-out clients and, ideally, a later outcome window → inspect the queue with editors.

The starter data is a single trailing-90-day snapshot. It supports exploration and a provisional proxy, but it does **not** by itself establish which pages will benefit from a refresh.

In [1]:
# A tiny framing check: the starter snapshot is content-item level, not daily-event level.
# The full data load and dataframe preview are shown in Section 4.
unit_of_analysis = "one pseudonymized content item in the 90-day snapshot"
task_type = "ranking / scoring"
print("Task type:", task_type)
print("Unit of analysis:", unit_of_analysis)

Task type: ranking / scoring
Unit of analysis: one pseudonymized content item in the 90-day snapshot


## 2. Target or proxy

**Ideal target (not present yet):** `future_decline_30d`, a label based on an observed decline in a clearly defined future 30-day window, using only features available before that window. This would connect the score to a future outcome and avoid using future information as input.

**Available starter proxy:** `is_declining_label = (trend_direction == "down")`. This label is a rule-derived summary of the current snapshot (`trend_direction` itself is calculated from `trend_pct`), not a separately observed future outcome. I can use it to sketch a first queue and understand the data, but it is a weak proxy and must not be described as “will decline.”

**Leakage guardrail:** never use `trend_direction`, `trend_pct`, or `is_declining_label` as model features when predicting that proxy. The first two directly define the label, and the third is the label itself. `content_id` and `client_id` are identifiers for grouping/splitting, not predictive features.

The code below creates the proxy explicitly and also sketches the future target as missing until a valid future window is available.

In [2]:
# The proxy is derived from the current snapshot, so it is only a framing aid.
# The future-looking target is intentionally missing: this CSV has no later outcome window.
print("Proxy definition: is_declining_label = (trend_direction == 'down')")
print("Ideal target sketch: future_decline_30d (currently unavailable in this snapshot)")

Proxy definition: is_declining_label = (trend_direction == 'down')
Ideal target sketch: future_decline_30d (currently unavailable in this snapshot)


## 3. Success metric

**Primary metric: Precision@K**, with K fixed before evaluation (for example, K = 100 pages an editor can review in a work cycle). It answers: “Of the top 100 pages we put in front of editors, what fraction match the chosen outcome label?” This aligns with a limited review capacity better than accuracy over all 30,000 items.

For an initial, clearly labeled proxy check, Precision@100 can be computed against `is_declining_label`; that only measures agreement with the current rule-derived proxy. It is **not** proof that the queue finds pages that will decline or that refreshing them will help.

For a defensible final evaluation, freeze the feature window, define a later observed outcome window, and report Precision@100 on held-out clients (and preferably a later time window), against both a simple rule baseline and any learned ranker. Also report how many eligible items exist and the proxy/outcome base rate. Do not pick K after looking at results.

In [3]:
# Confirm the chosen metric and evaluation caveat before any model is trained.
metric = "Precision@100"
print("Primary metric:", metric)
print("Interpretation: relevant items in the top 100 / 100 (when 100 eligible items exist)")
print("Current snapshot supports only a proxy evaluation, not a future-outcome claim.")

Primary metric: Precision@100
Interpretation: relevant items in the top 100 / 100 (when 100 eligible items exist)
Current snapshot supports only a proxy evaluation, not a future-outcome claim.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item** summarized by its trailing-90-day snapshot. The same client can contribute multiple content items, so rows are not independent client-level observations. Use `client_id` for grouped splits and never as a feature.

The slice below keeps fields useful for framing the refresh-opportunity lane. It shows both raw snapshot signals and the proxy label, with the label clearly separated from candidate features. `avg_position = 0` means “no data” in this dataset, not a true rank of zero. Rate columns such as `ctr` are stored as percentage values (for example, 0.76 means 0.76%), not fractions.

In [4]:
from pathlib import Path
import pandas as pd

# Locate the repository root whether the notebook is run from the root or work/notebooks.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
repo_root = next((candidate for candidate in candidates if (candidate / "data/raw/content_refresh_anonymized.csv").exists()), None)
if repo_root is None:
    # Colab may have opened the notebook without setting the repository as cwd.
    fallback = Path("/content/notebook-main")
    if (fallback / "data/raw/content_refresh_anonymized.csv").exists():
        repo_root = fallback
    else:
        raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv. Run this notebook from the repository root or clone the repository first.")

csv_path = repo_root / "data/raw/content_refresh_anonymized.csv"
data = pd.read_csv(csv_path)

# The lane's unit is one content item in the trailing-90-day snapshot.
proxy_data = data.copy()
proxy_data["is_declining_label"] = (proxy_data["trend_direction"] == "down").astype("int8")
proxy_data["future_decline_30d"] = pd.NA  # not observed in the supplied snapshot

lane_columns = [
    "content_id", "client_id", "content_type", "impressions_90d", "sessions_90d",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "trend_direction", "trend_pct", "is_declining_label", "future_decline_30d"
]
lane_df = proxy_data[lane_columns].copy()

print(f"Loaded: {csv_path.relative_to(repo_root)}")
print(f"Full starter dataset: {data.shape[0]:,} rows × {data.shape[1]} columns")
print("Unit of analysis: one content item (not one client and not one daily event).")
print("Proxy prevalence:", f"{lane_df['is_declining_label'].mean():.1%}")
display(lane_df.head(10))

print("Target/proxy sketch (first 8 rows):")
display(lane_df[["content_id", "trend_direction", "is_declining_label", "future_decline_30d"]].head(8))

Loaded: data/raw/content_refresh_anonymized.csv
Full starter dataset: 30,000 rows × 44 columns
Unit of analysis: one content item (not one client and not one daily event).
Proxy prevalence: 54.2%


,content_id,client_id,content_type,impressions_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,trend_direction,trend_pct,is_declining_label,future_decline_30d
0,content_304f48230142,client_f369cb89fc,keyword article,3803,17,187,20,0.76,10.6,down,-41.4,1,<NA>
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,9,445,25,0.05,20.3,down,-57.7,1,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,141,20,0.09,36.5,down,-60.9,1,<NA>
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,78,463,22,0.49,6.2,stable,-13.8,0,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,145,263,14,0.13,44.0,down,-34.7,1,<NA>
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,5,147,20,0.03,8.5,down,-38.9,1,<NA>
6,content_9a34b442b552,client_8722616204,keyword article,20,1,90,20,0.00,7.0,down,-92.3,1,<NA>
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,28,445,22,0.06,21.2,stable,0.6,0,<NA>
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,68,90,20,0.09,46.0,down,-58.8,1,<NA>
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,3,257,104,0.16,4.9,down,-29.2,1,<NA>


Target/proxy sketch (first 8 rows):


,content_id,trend_direction,is_declining_label,future_decline_30d
0,content_304f48230142,down,1,<NA>
1,content_a1fb4e703a9e,down,1,<NA>
2,content_9aa793d4d895,down,1,<NA>
3,content_331d6c4de07b,stable,0,<NA>
4,content_d99b7a2d90ca,down,1,<NA>
5,content_d4084a4bc775,down,1,<NA>
6,content_9a34b442b552,down,1,<NA>
7,content_a63219c6e95a,stable,0,<NA>


## 5. Why ML beats a fixed rule here

A single rule such as “review every page whose trend is down” is a useful baseline, but it ignores context: a page with substantial demand may deserve attention sooner than a low-demand page with the same direction; content age, recency of update, impressions, sessions, position, and content type may interact. A score could combine several signals and help order a queue when editor capacity is limited.

However, **ML has not earned its place yet**. A rule is cheaper, easier to explain, and may be good enough. The next experiment should compare a transparent baseline (for example, declining pages with meaningful impressions, sorted by demand) against a learned ranking method. The learned method is worth keeping only if it improves the predeclared metric on held-out clients / a later outcome window and gives useful, reviewable reason codes.

Important limitations:
- The supplied snapshot does not contain an observed future 30-day label, so it cannot yet validate future decline prediction honestly.
- A proxy-based score may just reproduce the rule used to define the proxy.
- A high score is not evidence that refreshing a page causes recovery. Causal claims need an experiment or another credible causal design.
- The output supports a human decision; it should not automatically refresh, prune, or rewrite content.

In [5]:
# Show how the current proxy is distributed, without treating it as future truth.
proxy_counts = lane_df["is_declining_label"].value_counts().rename(index={0: "not labeled down", 1: "labeled down"})
print("Proxy-label counts:")
display(proxy_counts.to_frame("content_items"))

# A compact unit-of-analysis check: content IDs should identify individual rows in this snapshot.
print("Rows:", len(lane_df))
print("Unique content IDs:", lane_df["content_id"].nunique())
print("Unique clients:", lane_df["client_id"].nunique())
print("Duplicate content IDs:", int(lane_df["content_id"].duplicated().sum()))

Proxy-label counts:


,content_items
is_declining_label,
labeled down,16262
not labeled down,13738


Rows: 30000
Unique content IDs: 30000
Unique clients: 32
Duplicate content IDs: 0


## Self-check

- [x] Named the lane and task type: refresh/content opportunity ranking or scoring.
- [x] Named the decision, user, action, and cost of a wrong call.
- [x] Distinguished an ideal future observed target from the available rule-derived proxy.
- [x] Named Precision@100 before training and explained its limits on proxy labels.
- [x] Loaded the starter dataset and displayed a real content-item dataframe plus target sketch.
- [x] Explained why a multi-signal score might help, while requiring it to beat a simple rule baseline.
- [x] Kept the claims descriptive and decision-support only; no causal or future-prediction claim is made.
- [ ] Commit this executed notebook to `work/notebooks/w02_ml_task_framing.ipynb` and submit the repository URL on the assignment card.

**My framing in one sentence:** For editors choosing which pages to review first, I will investigate a content-item ranking score and evaluate its top-K precision against a clearly defined outcome; the current `is_declining_label` is only a rule-derived proxy, so a future-outcome window is required before claiming predictive value.